# Retail Demand Forecasting Ops

End-to-end forecasting system with classical baselines plus LazyPredict, manual top-3, FLAML, and PyCaret tracks.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_prep import load_rossmann_data, merge_train_store, clean_sales_data, time_split
from src.features import engineer_calendar_features, add_lag_rolling_features, build_supervised_dataset, build_preprocessor, prepare_matrices
from src.forecast_models import (
    run_classical_baselines,
    run_lazypredict_discovery,
    select_top3_eligible_families,
    run_manual_engineering_track,
    run_flaml_track,
    run_pycaret_track,
    build_leaderboard,
    create_ops_signals,
)
from src.backtest import rolling_origin_backtest, summarize_backtest

SEED = 42
PROJECT_NAME = 'retail-demand-forecasting-ops'
PROJECT_ROOT = Path('.')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'rossmann'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## 1) Business Problem and Success Criteria

Forecast store-level daily sales to improve staffing and replenishment planning.

Success criteria:
- Low sMAPE (primary), MAE (secondary), RMSE (tertiary)
- Stable performance in rolling backtests
- Actionable operational signals

## 2) Dataset Access and Data Dictionary

Dataset: Rossmann Store Sales.

Core files: `train.csv`, `store.csv`, `test.csv`.

In [2]:
data = load_rossmann_data(RAW_DIR, sample_store_frac=0.08, random_state=SEED)
{k: v.shape for k, v in data.items()}

/home/ahmad/AI/Projects/retail-demand-forecasting-ops/src/data_prep.py:12: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path)


{'train': (80893, 9), 'test': (3072, 8), 'store': (89, 10)}

## 3) Data Cleaning and Leakage Checks

- Merge train and store data
- Ensure chronological ordering
- Keep only historical lag-safe features

In [3]:
merged = merge_train_store(data['train'], data['store'])
clean = clean_sales_data(merged)
clean.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,32,2,2013-01-01,0,0,0,0,a,1,a,a,2910.0,NaN,NaN,1,45.0,2009.0,"Feb,May,Aug,Nov"
1,32,3,2013-01-02,3668,452,1,0,0,1,a,a,2910.0,NaN,NaN,1,45.0,2009.0,"Feb,May,Aug,Nov"
2,32,4,2013-01-03,2676,349,1,0,0,1,a,a,2910.0,NaN,NaN,1,45.0,2009.0,"Feb,May,Aug,Nov"
3,32,5,2013-01-04,3058,388,1,0,0,1,a,a,2910.0,NaN,NaN,1,45.0,2009.0,"Feb,May,Aug,Nov"
4,32,6,2013-01-05,1768,237,1,0,0,0,a,a,2910.0,NaN,NaN,1,45.0,2009.0,"Feb,May,Aug,Nov"


## 4) Feature Engineering

- Lag and rolling features
- Calendar, promo, holiday, and competition features

In [4]:
fe = engineer_calendar_features(clean)
fe = add_lag_rolling_features(fe, target_col='Sales')
supervised = build_supervised_dataset(fe, target_col='Sales')
supervised.shape

(78401, 36)

## 5) Validation Strategy

- Strict time-aware holdout split
- Rolling-origin backtest for stability checks
- Metrics prioritized by business impact

In [5]:
train_df, holdout_df = time_split(supervised, holdout_days=42)
train_df['Date'].min(), train_df['Date'].max(), holdout_df['Date'].min(), holdout_df['Date'].max()

(Timestamp('2013-01-29 00:00:00'),
 Timestamp('2015-06-19 00:00:00'),
 Timestamp('2015-06-20 00:00:00'),
 Timestamp('2015-07-31 00:00:00'))

In [6]:
preprocessor = build_preprocessor(train_df.drop(columns=['Sales', 'Date']))
X_train, X_holdout, y_train, y_holdout = prepare_matrices(train_df, holdout_df, 'Sales', preprocessor)
X_train.shape, X_holdout.shape

((74663, 45), (3738, 45))

## 6) Track 1: LazyPredict Discovery Lab

Run LazyPredict on supervised lag-feature table.

In [7]:
lazy_table = run_lazypredict_discovery(X_train, X_holdout, y_train, y_holdout)
lazy_table.head(15)

,Model,R-Squared,RMSE,Time Taken
0,CatBoostRegressor,0.984310,476.862027,1.248509
1,LGBMRegressor,0.979577,544.055955,0.107131
2,HistGradientBoostingRegressor,0.979285,547.939651,6.041203
3,XGBRegressor,0.977736,568.054465,2.358222
4,ExtraTreesRegressor,0.975290,598.437779,2.951837
5,RandomForestRegressor,0.971208,645.987459,6.624187
6,GradientBoostingRegressor,0.970766,650.927521,3.206813
7,BaggingRegressor,0.966307,698.809385,0.676456
8,PassiveAggressiveRegressor,0.945635,887.655548,0.053099
9,RANSACRegressor,0.945628,887.718507,0.361586


## 7) Selection of Top 3 Eligible Models

Only eligible model families proceed into manual engineering.

In [8]:
eligible_table, top3_families = select_top3_eligible_families(
    lazy_table, X_train, y_train, X_holdout, y_holdout, random_state=SEED
)
eligible_table, top3_families

(          lazy_model    family     smape         mae        rmse  \
 0       XGBRegressor   xgboost  0.213380  268.224752  397.623301   
 1  CatBoostRegressor  catboost  0.240201  284.525380  414.838032   
 2      LGBMRegressor  lightgbm  0.241544  276.484496  401.096943   
 
    train_time_sec  eligible eligibility_note  
 0      176.098631      True         eligible  
 1        4.861554      True         eligible  
 2      356.386312      True         eligible  ,
 ['catboost', 'lightgbm', 'xgboost'])

## 8) Track 2: Manual Engineering Lab

Manual implementation of only the selected top-3 families.

In [9]:
manual_results, manual_models, manual_preds = run_manual_engineering_track(
    top3_families, X_train, y_train, X_holdout, y_holdout, random_state=SEED
)
manual_results[['model_name', 'smape', 'mae', 'rmse']].sort_values('smape')

,model_name,smape,mae,rmse
0,xgboost,0.213380,268.224752,397.623301
1,catboost,0.240201,284.525380,414.838032
2,lightgbm,0.241544,276.484496,401.096943


## 9) Track 3: FLAML Optimization Lab

Budget-aware AutoML search on lag-feature regression task.

In [10]:
flaml_result = run_flaml_track(
    X_train=X_train,
    y_train=y_train,
    X_holdout=X_holdout,
    y_holdout=y_holdout,
    time_budget=300,
    random_state=SEED,
)
flaml_result

[flaml.automl.logger: 06-11 19:00:54] {2375} INFO - task = regression


[flaml.automl.logger: 06-11 19:00:54] {2386} INFO - Evaluation method: cv


[flaml.automl.logger: 06-11 19:00:54] {2489} INFO - Minimizing error metric: mae


[flaml.automl.logger: 06-11 19:00:54] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'xgboost', 'rf', 'extra_tree', 'xgb_limitdepth']


[flaml.automl.logger: 06-11 19:00:54] {2911} INFO - iteration 0, current learner lgbm


...

[flaml.automl.logger: 06-11 19:06:01] {3359} INFO - retrain xgboost for 6.9s


[flaml.automl.logger: 06-11 19:06:01] {3362} INFO - retrained model: XGBRegressor(base_score=None, booster=None, callbacks=[],
             colsample_bylevel=0.9602017468771757, colsample_bynode=None,
             colsample_bytree=0.9993271961638156, device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=None,
             grow_policy='lossguide', importance_type=None,
             interaction_constraints=None, learning_rate=0.20283799772488423,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=0, max_leaves=44,
             min_child_weight=0.22832695926526178, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=299,
             n_jobs=-1, num_parallel_tree=None, random_state=None, ...)


[flaml.automl.logger: 06-11 19:06:01] {2636} INFO - fit succeeded


[flaml.automl.logger: 06-11 19:06:01] {2637} INFO - Time taken to find the best model: 109.79451251029968


{'model_name': 'xgboost',
 'library_source': 'flaml',
 'smape': 0.2763714956279103,
 'mae': 287.527133323901,
 'rmse': 421.8045997351679,
 'train_time_sec': 306.65105553300236,
 'infer_latency_ms': nan,
 'p95_latency_ms': nan,
 'interpretability_note': 'FLAML budget-aware challenger',
 'best_config': {'n_estimators': 299,
  'max_leaves': 44,
  'min_child_weight': 0.22832695926526178,
  'learning_rate': 0.20283799772488423,
  'subsample': 0.8406285097972522,
  'colsample_bylevel': 0.9602017468771757,
  'colsample_bytree': 0.9993271961638156,
  'reg_alpha': 0.0014585172191691575,
  'reg_lambda': 9.711377824359996},
 'best_loss': 264.862027088784,
 'time_budget': 300,
 'predictions': array([2201.9077,    0.    , 3639.368 , ..., 6138.652 , 5728.808 ,
        6177.364 ], dtype=float32)}

## 10) Track 4: PyCaret Experiment Lab

PyCaret compare/tune/finalize workflow on regression experiment.

In [11]:
pycaret_train = train_df.drop(columns=['Date']).copy()
pycaret_holdout = holdout_df.drop(columns=['Date']).copy()
pycaret_result = run_pycaret_track(
    train_df=pycaret_train,
    holdout_df=pycaret_holdout,
    target_col='Sales',
    session_id=SEED,
    model_output_path=ARTIFACT_DIR / 'pycaret_retail_forecasting',
)
pycaret_result

{'model_name': 'pycaret_failed',
 'library_source': 'pycaret',
 'smape': nan,
 'mae': nan,
 'rmse': nan,
 'train_time_sec': nan,
 'infer_latency_ms': nan,
 'p95_latency_ms': nan,
 'interpretability_note': "PyCaret unavailable or failed: ('Pycaret only supports python 3.9, 3.10, 3.11. Your actual Python version: ', sys.version_info(major=3, minor=12, micro=10, releaselevel='final', serial=0), 'Please DOWNGRADE your Python version.')",
 'predictions': array([], dtype=float64),
 'status': 'failed'}

## 11) Unified Leaderboard and Final Model Ranking

Compare classical baselines + manual top-3 + best FLAML + best PyCaret.

In [12]:
classical_baselines, holdout_with_baselines = run_classical_baselines(train_df, holdout_df)
leaderboard = build_leaderboard(
    project_name=PROJECT_NAME,
    lazy_results=eligible_table,
    manual_results=manual_results,
    flaml_result=flaml_result,
    pycaret_result=pycaret_result,
    classical_baselines=classical_baselines,
)
leaderboard.head(10)

19:06:02 - cmdstanpy - INFO - Chain [1] start processing


19:06:02 - cmdstanpy - INFO - Chain [1] done processing


,project_name,task_type,library_source,model_name,cv_metric_mean,cv_metric_std,holdout_primary_metric,holdout_secondary_metric,holdout_tertiary_metric,calibration_metric,train_time_sec,infer_latency_ms,p95_latency_ms,model_size_mb,retrain_time_sec,interpretability_note,rank_score,final_rank
0,retail-demand-forecasting-ops,time_series_regression,lazypredict,xgboost,NaN,NaN,0.213380,268.224752,397.623301,NaN,176.098631,NaN,NaN,NaN,176.098631,eligible,0.689356,1
1,retail-demand-forecasting-ops,time_series_regression,manual,xgboost,NaN,NaN,0.213380,268.224752,397.623301,NaN,10.292310,0.815481,1.923312,NaN,10.292310,Manual lag-feature model: xgboost,0.689356,2
2,retail-demand-forecasting-ops,time_series_regression,lazypredict,catboost,NaN,NaN,0.240201,284.525380,414.838032,NaN,4.861554,NaN,NaN,NaN,4.861554,eligible,0.653495,3
3,retail-demand-forecasting-ops,time_series_regression,manual,catboost,NaN,NaN,0.240201,284.525380,414.838032,NaN,4.786789,0.367267,0.433698,NaN,4.786789,Manual lag-feature model: catboost,0.653495,4
4,retail-demand-forecasting-ops,time_series_regression,lazypredict,lightgbm,NaN,NaN,0.241544,276.484496,401.096943,NaN,356.386312,NaN,NaN,NaN,356.386312,eligible,0.653233,5
5,retail-demand-forecasting-ops,time_series_regression,manual,lightgbm,NaN,NaN,0.241544,276.484496,401.096943,NaN,265.348224,3.059658,14.249887,NaN,265.348224,Manual lag-feature model: lightgbm,0.653233,6
6,retail-demand-forecasting-ops,time_series_regression,classical_baseline,seasonal_naive,NaN,NaN,0.155931,1100.508860,1576.161479,NaN,NaN,NaN,NaN,NaN,NaN,Store/day-of-week seasonal baseline,0.625181,7
7,retail-demand-forecasting-ops,time_series_regression,flaml,xgboost,NaN,NaN,0.276371,287.527133,421.804600,NaN,306.651056,NaN,NaN,NaN,306.651056,FLAML budget-aware challenger,0.607780,8
8,retail-demand-forecasting-ops,time_series_regression,classical_baseline,prophet,NaN,NaN,0.466287,1296.153654,1863.274318,NaN,NaN,NaN,NaN,NaN,NaN,Aggregate Prophet reference,0.205767,9
9,retail-demand-forecasting-ops,time_series_regression,classical_baseline,sarima,NaN,NaN,0.481099,1655.173915,2107.481600,NaN,NaN,NaN,NaN,NaN,NaN,Aggregate SARIMA reference,0.139930,10


In [13]:
leaderboard.to_csv(ARTIFACT_DIR / 'leaderboard_retail_forecasting.csv', index=False)
leaderboard[['model_name', 'library_source', 'holdout_primary_metric', 'rank_score', 'final_rank']].head(10)

,model_name,library_source,holdout_primary_metric,rank_score,final_rank
0,xgboost,lazypredict,0.213380,0.689356,1
1,xgboost,manual,0.213380,0.689356,2
2,catboost,lazypredict,0.240201,0.653495,3
3,catboost,manual,0.240201,0.653495,4
4,lightgbm,lazypredict,0.241544,0.653233,5
5,lightgbm,manual,0.241544,0.653233,6
6,seasonal_naive,classical_baseline,0.155931,0.625181,7
7,xgboost,flaml,0.276371,0.607780,8
8,prophet,classical_baseline,0.466287,0.205767,9
9,sarima,classical_baseline,0.481099,0.139930,10


## 12) Business Recommendation

After execution, explain winner rationale, safer second-best scenario, and key rejected tradeoff.

## 13) Inference / Deployment Path

Generate operations planning signals from selected winner predictions.

In [14]:
winner = leaderboard.sort_values('final_rank').iloc[0]
if winner['library_source'] == 'manual' and winner['model_name'] in manual_preds:
    prediction = manual_preds[winner['model_name']]
elif winner['library_source'] == 'flaml':
    prediction = flaml_result.get('predictions', np.array([]))
elif winner['library_source'] == 'pycaret':
    prediction = pycaret_result.get('predictions', np.array([]))
else:
    prediction = holdout_with_baselines.get('pred_naive', pd.Series(dtype=float)).to_numpy()

ops_base = holdout_df[['Date', 'Store', 'Sales']].reset_index(drop=True).copy()
ops_base['prediction'] = prediction[:len(ops_base)] if len(prediction) else np.nan
ops_signals = create_ops_signals(ops_base, pred_col='prediction')
ops_signals.to_csv(ARTIFACT_DIR / 'ops_signals.csv', index=False)
ops_signals.head()

,Date,Store,Sales,prediction,staffing_proxy,replenishment_signal,demand_spike_alert
0,2015-06-20,32,2215,5085,5,normal,False
1,2015-06-21,32,0,5085,5,normal,False
2,2015-06-22,32,3619,5085,5,normal,False
3,2015-06-23,32,3752,5085,5,normal,False
4,2015-06-24,32,3409,5085,5,normal,False


## 14) Monitoring / Drift / Retraining Plan

Track sMAPE drift by store cluster, holiday error spikes, and alert precision. Retrain monthly or on sustained drift.

## 15) Limitations and Next Steps

- Add richer exogenous signals (weather/events)
- Add probabilistic intervals
- Add scenario simulation for promo planning

In [15]:
def naive_predictor(train_slice, valid_slice):
    last = train_slice.sort_values('Date').groupby('Store')['Sales'].last()
    return valid_slice['Store'].map(last).fillna(train_slice['Sales'].mean()).to_numpy()

backtest_df = rolling_origin_backtest(supervised, predictor_fn=naive_predictor, min_train_days=180, horizon_days=14, step_days=14)
backtest_summary = summarize_backtest(backtest_df)
backtest_summary.to_csv(ARTIFACT_DIR / 'backtest_summary.csv', index=False)
backtest_summary

,metric,mean,std
0,smape,1.620333,0.084602
1,mae,5790.358760,467.757351
2,rmse,7042.491936,625.868692
